# 공개 가능한 공간 3D 복원 (MapAnything, 코랩 T4)

**먼저:** 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU** 를 고른 뒤 **런타임 → 모두 실행**.

중간에 파일 고르는 창이 뜨면 `frames.zip` 을 올립니다. 마지막 셀이 `points.npz`, `metrics.json` 을 내려받게 합니다.
PLY 만들기와 형식 변환은 내 컴퓨터에서 합니다.

In [ ]:
import torch, time, sys, os
T0 = time.time()
assert torch.cuda.is_available(), "GPU 가 없습니다. 런타임 유형을 T4 GPU 로 바꾸고 다시 실행하세요."
p = torch.cuda.get_device_properties(0); cap = torch.cuda.get_device_capability()
print(f"{p.name} / {p.total_memory/1e9:.1f}GB / 계산 능력 {cap[0]}.{cap[1]} / bf16 {cap[0] >= 8}")

## 설치 (torch 는 건드리지 않음)

In [ ]:
%%capture
!git clone -q --depth 1 https://github.com/facebookresearch/map-anything.git
!cd map-anything && pip install -q -e .

In [ ]:
sys.path.insert(0, '/content/map-anything')
import mapanything
print("설치까지", round(time.time()-T0), "초")

## 모델 준비 (4.91GB, 약 3분) — 영상 올리기 전에 먼저 받음

In [ ]:
from mapanything.models import MapAnything
from mapanything.utils.image import load_images
t = time.time()
model = MapAnything.from_pretrained("facebook/map-anything-apache").to("cuda").eval()
T_MODEL = time.time() - t
print(f"모델 준비 {T_MODEL:.0f}초")

## 프레임 올리기 — `frames.zip` 선택

In [ ]:
import zipfile, glob
from google.colab import files
up = files.upload()
ZIP = os.path.join('/content', list(up.keys())[0])
os.makedirs('/content/frames', exist_ok=True)
for f in glob.glob('/content/frames/*'): os.remove(f)
with zipfile.ZipFile(ZIP) as z:
    for n in z.namelist():
        if n.lower().endswith('.jpg'):
            with open(os.path.join('/content/frames', os.path.basename(n)), 'wb') as fo:
                fo.write(z.read(n))
paths = sorted(glob.glob('/content/frames/*.jpg'))
print(len(paths), "장")

## 복원 (T4 는 반드시 fp16)

In [ ]:
t = time.time()
views = load_images(paths)
IMG_SHAPE = tuple(views[0]['img'].shape[-2:])
print("모델 입력 크기", IMG_SHAPE)
torch.cuda.reset_peak_memory_stats()
t = time.time()
pred = model.infer(views, memory_efficient_inference=True, minibatch_size=1,
                   use_amp=True, amp_dtype="fp16",
                   apply_mask=True, mask_edges=True, apply_confidence_mask=False)
torch.cuda.synchronize()
T_INFER = time.time() - t
PEAK = torch.cuda.max_memory_allocated() / 1e9
print(f"복원 {T_INFER:.1f}초, 최대 GPU 메모리 {PEAK:.2f}GB")

## 점 정리, 카메라 퍼짐 비

In [ ]:
import numpy as np, json
pts, cols, cams, per_view = [], [], [], []
for v in pred:
    m = v['mask'][0, ..., 0].cpu().numpy()
    pts.append(v['pts3d'][0].cpu().numpy()[m])
    cols.append((v['img_no_norm'][0].cpu().numpy() * 255).clip(0, 255).astype(np.uint8)[m])
    cams.append(v['camera_poses'][0].cpu().numpy())
    per_view.append(int(m.sum()))
xyz = np.concatenate(pts).astype(np.float32); rgb = np.concatenate(cols); cams = np.stack(cams).astype(np.float32)
view_id = np.concatenate([np.full(n, i, np.uint8) for i, n in enumerate(per_view)])
N_RAW = len(xyz)
lo = np.percentile(xyz, 0.5, axis=0); hi = np.percentile(xyz, 99.5, axis=0)
keep = np.all((xyz > lo) & (xyz < hi), axis=1)
xyz, rgb, view_id = xyz[keep], rgb[keep], view_id[keep]
cpos = cams[:, :3, 3]
spread = float(np.linalg.norm(cpos.max(0) - cpos.min(0)))
diag = float(np.linalg.norm(xyz.max(0) - xyz.min(0)))
path_len = float(np.linalg.norm(np.diff(cpos, axis=0), axis=1).sum())
print(f"점 {N_RAW:,} -> 튄 점 제거 후 {len(xyz):,}")
print(f"카메라 퍼짐 {spread:.3f} / 장면 대각선 {diag:.3f} = {spread/diag:.4f}  (0.1 보다 커야 정상)")
metrics = dict(gpu=p.name, frames=len(paths), model_input_hw=list(IMG_SHAPE),
               t_model_s=round(T_MODEL, 1), t_infer_s=round(T_INFER, 2), peak_gpu_gb=round(PEAK, 2),
               points_raw=N_RAW, points_kept=int(len(xyz)), points_per_view=per_view,
               cam_spread=spread, scene_diag=diag, spread_ratio=spread/diag, cam_path_len=path_len)
np.savez_compressed('/content/points.npz', xyz=xyz, rgb=rgb, cams=cams, view_id=view_id)
json.dump(metrics, open('/content/metrics.json', 'w'), indent=1, ensure_ascii=False)
print(f"points.npz {os.path.getsize('/content/points.npz')/1e6:.1f}MB")

## 내려받기 (브라우저가 두 파일 저장 허용을 물으면 허용)

In [ ]:
files.download('/content/metrics.json')
files.download('/content/points.npz')
print("전 과정", round(time.time()-T0), "초 — 다 받았으면 런타임 → 런타임 연결 해제 및 삭제")